# OMERO @SURF 2

In [1]:
import ezomero
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Connect
from getpass import getpass

server = 'omero1.fair-omero-lu.src.surf-hosted.nl'
user = 'nlbi'
group = "training"
password = getpass('OMERO password: ')

# Preferred: encrypted Ice connection on port 4064.
# Fallback: unencrypted Ice connection on port 4063. Some corporate networks
# (firewalls / endpoint-security agents doing TLS inspection) silently reset the
# TLS handshake on non-standard ports, which shows up as
# Ice.ConnectionLostException. Port 4063 speaks plain Ice and usually survives.
conn = None
for protocol, port in (('ssl', 4064), ('tcp', 4063)):
    try:
        conn = ezomero.connect(host=f'{protocol}://{server}:{port}', port=port,
                               user=user, password=password, group=group,
                               secure=(protocol == 'ssl'))
    except Exception as e:
        print(f'{protocol}://{server}:{port} failed: {type(e).__name__}: {e}')
        continue
    if conn is not None:
        print(f'Connected over {protocol} on port {port}'
              + ('' if protocol == 'ssl' else '  (unencrypted - fallback)'))
        break

if conn is None:
    raise RuntimeError('Could not connect to OMERO on either port 4064 or 4063')


OMERO password:  ········


Connected over ssl on port 4064


In [3]:
#List screens
screen_ids = ezomero.get_screen_ids(conn)    
for sid in screen_ids:
    screen_name = conn.getObject("Screen", sid).getName()
    print(f'Screen ID: {sid} | Name: {screen_name}')

Screen ID: 1 | Name: nuclei_segmentation


In [4]:
plate_ids = ezomero.get_plate_ids(conn, screen=1)
print(f'Plate IDs: {plate_ids} ')
well_ids = ezomero.get_well_ids(conn,plate=plate_ids[0])
print(f'Well IDs: {well_ids} ')

Plate IDs: [1] 
Well IDs: [24, 2, 7, 9, 3, 19, 14, 15, 1, 22, 21, 20, 13, 23, 10, 18, 4, 16, 11, 12, 17, 5, 8, 6] 


In [5]:
plate = conn.getObject("Plate", plate_ids[0])

# listChildren returns wells unsorted, so sort by grid position
wells = sorted(plate.listChildren(), key=lambda w: (w.getRow(), w.getColumn()))
print(f"{len(wells)} wells")
for well in wells:
    print(
        well.getWellPos(),
        "fields:", len(list(well.listChildren())),
        "annotations:", [annotation.getValue() for annotation in well.listAnnotations()]
    )

24 wells
B2 fields: 4 annotations: [[('cellline', 'Hs578t'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
B3 fields: 4 annotations: [[('cellline', 'Hs578t'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
B4 fields: 4 annotations: [[('cellline', 'MDAMB231'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
B5 fields: 4 annotations: [[('cellline', 'MDAMB231'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
B6 fields: 4 annotations: [[('cellline', 'HCC38'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
B7 fields: 4 annotations: [[('cellline', 'HCC38'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'DMSO')]]
C2 fields: 4 annotations: [[('cellline', 'Hs578t'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'flavopiridol')]]
C3 fields: 4 annotations: [[('cellline', 'Hs578t'), ('channel1', 'brightfield'), ('channel2'

In [6]:
#get dimensions of one image
well = wells[10]
image = well.getImage(0)          # first field of this well

print(f"{well.getWellPos()} — {image.getName()} (ID {image.getId()})")
print(f"X={image.getSizeX()} Y={image.getSizeY()} Z={image.getSizeZ()} "
      f"C={image.getSizeC()} T={image.getSizeT()} dtype={image.getPixelsType()}")
print([annotation.getValue() for annotation in well.listAnnotations()])

C6 — 20251111_morphology_XW_plate1406.companion.ome [Well C6, Field #1] (ID 111)
X=1408 Y=1040 Z=1 C=2 T=37 dtype=uint16
[[('cellline', 'HCC38'), ('channel1', 'brightfield'), ('channel2', 'none'), ('treatment', 'flavopiridol')]]


In [7]:
z = image.getSizeZ() // 2
c, t = 0, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

plane shape: (1040, 1408) uint16 range: 0 773


In [8]:
z = image.getSizeZ() // 2
c, t = 1, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

plane shape: (1040, 1408) uint16 range: 3 255


In [9]:
#load one plane
z = image.getSizeZ() // 2
c, t = 1, 0

plane = image.getPrimaryPixels().getPlane(theZ=z, theC=c, theT=t)
print("plane shape:", plane.shape, plane.dtype, "range:", plane.min(), plane.max())

plane shape: (1040, 1408) uint16 range: 3 255


In [10]:
plane_norm = plane / plane.max()
# plt.figure(figsize=(6, 6))
# plt.imshow(plane_norm, cmap="gray")
# plt.title(f"{well.getWellPos()} — z{z} c{c} t{t}")
# plt.axis("off")
# plt.show()

In [11]:
plane_norm8 = (plane_norm * 255).astype(np.uint8)
output_image = np.expand_dims(plane_norm8, axis=(0, 1, 2))  # insert TCZ axes

# Put the connection back in a real group. OMERO refuses to create new
# objects while the group context is -1 ("all groups").
group_id = conn.getEventContext().groupId
ezomero.set_group(conn, group_id)
dataset_id = ezomero.post_dataset(conn, "Test Dataset")

image_id = ezomero.post_image(conn, output_image, "test", dim_order="tczyx", dataset_id=dataset_id)
print('New image created:', image_id)

New image created: 263
